In [ ]:
import pandas as pd

import numpy as np

def parse_hand_location(filepath):
    """
    Parses Hand_Location.txt (Space delimited).
    Returns: DataFrame [Time, Accuracy, Lat, Lon, Alt]
    """
    print(f"Processing {filepath}...")
    try:
        df = pd.read_csv(filepath, sep=r'\s+', header=None, 
                         usecols=[0, 3, 4, 5, 6], 
                         names=['Time', 'Accuracy', 'Lat', 'Lon', 'Alt'])
        return df
    except Exception as e:
        print(f"Error parsing Location: {e}")
        return pd.DataFrame()

def parse_hand_cells(filepath):
    """
    Parses Hand_Cells.txt (Variable length).
    Extracts secondary identifiers (PCI/PSC/ARFCN) so neighbor cells 
    with CID=INT_MAX are not lost.
    
    Field layout per cell type:
      LTE   (10 fields): TYPE reg CID MCC MNC PCI TAC  signal RSRP TA
      WCDMA (10 fields): TYPE reg CID LAC MCC MNC PSC  signal RSCP ASU
      GSM   ( 9 fields): TYPE reg CID LAC MCC MNC BSIC RSSI   ASU
    
    Returns: DataFrame [Time, Type, CID, Unique_ID, RSSI]
      - Unique_ID = "{Type}_{secondary_id}" e.g. LTE_144, WCDMA_158, GSM_8
      - RSSI = signal strength in dBm (RSRP for LTE, RSCP for WCDMA, RSSI for GSM)
    """
    print(f"Processing {filepath}...")
    parsed_data = []
    
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if not parts or len(parts) < 4:
                continue
            try:
                timestamp = int(parts[0])
            except ValueError:
                continue
            
            idx = 4  # entries start after Time, Ignore, Ignore, Count
            
            while idx < len(parts):
                cell_type = parts[idx]
                
                if cell_type == 'LTE':
                    # LTE: TYPE reg CID MCC MNC PCI TAC signal RSRP TA (10 fields)
                    if idx + 9 < len(parts):
                        cid   = parts[idx + 2]
                        pci   = parts[idx + 5]   # Physical Cell ID (0-503)
                        rsrp  = parts[idx + 8]   # RSRP in dBm
                        uid   = f"LTE_{pci}"
                        parsed_data.append({
                            'Time': timestamp, 'Type': 'LTE',
                            'CID': cid, 'Unique_ID': uid, 'RSSI': int(rsrp)
                        })
                    idx += 10
                    
                elif cell_type == 'GSM':
                    # GSM: TYPE reg CID LAC MCC MNC BSIC RSSI ASU (9 fields)
                    if idx + 8 < len(parts):
                        cid   = parts[idx + 2]
                        bsic  = parts[idx + 6]   # BSIC / ARFCN
                        rssi  = parts[idx + 7]   # RSSI in dBm
                        uid   = f"GSM_{bsic}"
                        parsed_data.append({
                            'Time': timestamp, 'Type': 'GSM',
                            'CID': cid, 'Unique_ID': uid, 'RSSI': int(rssi)
                        })
                    idx += 9
                    
                elif cell_type == 'WCDMA':
                    # WCDMA: TYPE reg CID LAC MCC MNC PSC signal RSCP ASU (10 fields)
                    if idx + 9 < len(parts):
                        cid   = parts[idx + 2]
                        psc   = parts[idx + 6]   # Primary Scrambling Code
                        rscp  = parts[idx + 8]   # RSCP in dBm
                        uid   = f"WCDMA_{psc}"
                        parsed_data.append({
                            'Time': timestamp, 'Type': 'WCDMA',
                            'CID': cid, 'Unique_ID': uid, 'RSSI': int(rscp)
                        })
                    idx += 10
                else:
                    break
                    
    return pd.DataFrame(parsed_data)

def parse_hand_wifi(filepath):
    """
    Parses Hand_WiFi.txt (Semicolon delimited).
    Returns: DataFrame [Time, BSSID, SSID, RSSI]
    """
    print(f"Processing {filepath}...")
    parsed_data = []
    
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split(';')
            if len(parts) < 4:
                continue
            try:
                timestamp = int(parts[0])
                num_entries = int(parts[3])
            except ValueError:
                continue
            
            current_idx = 4
            for _ in range(num_entries):
                if current_idx + 4 < len(parts):
                    bssid = parts[current_idx]
                    ssid  = parts[current_idx + 1]
                    rssi  = parts[current_idx + 2]
                    parsed_data.append({
                        'Time': timestamp, 'BSSID': bssid,
                        'SSID': ssid, 'RSSI': int(rssi)
                    })
                    current_idx += 5
                else:
                    break

    return pd.DataFrame(parsed_data)

In [ ]:
from pathlib import Path

# --- Resolve dataset folder automatically (keep User1/220617 only) ---
base_candidates = [
    Path('selected'),
    Path('SHLDataset_preview_v2/User1/220617'),
]

data_dir = None
for c in base_candidates:
    if (c / 'Hand_Location.txt').exists() and (c / 'Hand_Cells.txt').exists() and (c / 'Hand_WiFi.txt').exists():
        data_dir = c
        break

if data_dir is None:
    raise FileNotFoundError(
        "Could not find Hand_* files. Expected either:\n"
        "  - selected/Hand_*.txt\n"
        "  - SHLDataset_preview_v2/User1/220617/Hand_*.txt"
    )

print(f"Using data directory: {data_dir}")

# --- Parse the three data sources ---
df_loc   = parse_hand_location(str(data_dir / 'Hand_Location.txt'))
df_cells = parse_hand_cells(str(data_dir / 'Hand_Cells.txt'))
df_wifi  = parse_hand_wifi(str(data_dir / 'Hand_WiFi.txt'))

print(f"\nLocation : {len(df_loc):,} rows")
print(f"Cells    : {len(df_cells):,} rows  ({df_cells['Unique_ID'].nunique()} unique IDs)")
print(f"WiFi     : {len(df_wifi):,} rows   ({df_wifi['BSSID'].nunique()} unique BSSIDs)")

# --- Quick sanity check ---
print("\nCell type distribution:")
print(df_cells['Type'].value_counts())
print("\nTop 10 cell IDs by frequency:")
print(df_cells['Unique_ID'].value_counts().head(10))

In [ ]:
from matplotlib import pyplot as plt


plt.figure(figsize=(10, 8))
# Longitude on X, Latitude on Y is standard map orientation
sc = plt.scatter(df_loc['Lon'], df_loc['Lat'], c=df_loc['Time'], cmap='plasma', s=2, alpha=0.5)
plt.colorbar(sc, label='Time (Trajectory)')
plt.title('Ground Truth GPS Trajectory (Hand Position)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.axis('equal') # Keep aspect ratio correct for a map

# Invert X axis if needed? No, standard Lon increases to Right.
# Check if we need to filter outliers (0,0 coordinates)
# The sample showed 50.84 lat, -0.13 lon (Brighton, UK area), which looks valid.

plt.tight_layout()
plt.savefig('gps_groundtruth_map.png')
plt.show()

In [ ]:
## -- Step 2: Build Combined Fingerprint Matrices (Cells + WiFi) --

def create_combined_channel_matrices(df_cells, df_wifi, time_bin='5s',
                                     cell_noise=-120, wifi_noise=-100,
                                     min_appearances=10):
    """
    Build three aligned matrices:
      X_val  : scaled RSSI in [0,1], NaN for missing observations
      X_mask : 1 if observed, 0 if missing
      X_model: dense matrix for NN input (NaN -> 0 after scaling)
    """
    # -- prepare cell data --
    df_c = df_cells.copy()
    df_c['Time_dt'] = pd.to_datetime(df_c['Time'], unit='ms')
    df_c['TimeBin'] = df_c['Time_dt'].dt.floor(time_bin)

    # -- prepare WiFi data --
    df_w = df_wifi.copy()
    df_w['Time_dt'] = pd.to_datetime(df_w['Time'], unit='ms')
    df_w['TimeBin'] = df_w['Time_dt'].dt.floor(time_bin)
    df_w['Unique_ID'] = 'WiFi_' + df_w['BSSID']

    # -- filter rare transmitters --
    cell_counts = df_c.groupby('Unique_ID')['TimeBin'].nunique()
    wifi_counts = df_w.groupby('Unique_ID')['TimeBin'].nunique()

    valid_cells = cell_counts[cell_counts >= min_appearances].index
    valid_wifis = wifi_counts[wifi_counts >= min_appearances].index

    df_c = df_c[df_c['Unique_ID'].isin(valid_cells)]
    df_w = df_w[df_w['Unique_ID'].isin(valid_wifis)]

    print(f"Cells : {len(valid_cells)} transmitters kept  "
          f"({len(cell_counts) - len(valid_cells)} rare dropped)")
    print(f"WiFi  : {len(valid_wifis)} APs kept  "
          f"({len(wifi_counts) - len(valid_wifis)} rare dropped)")

    # -- pivot each source --
    piv_c = df_c.pivot_table(index='TimeBin', columns='Unique_ID',
                             values='RSSI', aggfunc='mean')
    piv_w = df_w.pivot_table(index='TimeBin', columns='Unique_ID',
                             values='RSSI', aggfunc='mean')

    # -- keep NaN for missing; do NOT hard-fill before distance computation --
    X_val = piv_c.join(piv_w, how='outer')
    X_val = X_val.reindex(sorted(X_val.columns), axis=1)

    # source-specific scaling on observed values only
    cell_cols = [c for c in X_val.columns if not c.startswith('WiFi_')]
    wifi_cols = [c for c in X_val.columns if c.startswith('WiFi_')]

    cell_max, wifi_max = -50, -30
    X_val[cell_cols] = ((X_val[cell_cols] - cell_noise)
                        / (cell_max - cell_noise)).clip(0, 1)
    X_val[wifi_cols] = ((X_val[wifi_cols] - wifi_noise)
                        / (wifi_max - wifi_noise)).clip(0, 1)

    # mask and dense model matrix
    X_mask = (~X_val.isna()).astype(np.float32)
    X_model = X_val.fillna(0.0).astype(np.float32)

    return X_val, X_mask, X_model

# -- build matrices --
X_val, X_mask, X = create_combined_channel_matrices(df_cells, df_wifi, time_bin='5s')
print(f"\nFinal matrix shape: {X.shape}  (time bins x features)")
print(f"  Cell features : {sum(not c.startswith('WiFi_') for c in X.columns)}")
print(f"  WiFi features : {sum(c.startswith('WiFi_') for c in X.columns)}")
print(f"  Time range    : {X.index.min()} -> {X.index.max()}")
print(f"  Observed ratio: {X_mask.values.mean():.3f}")

In [ ]:
## -- Step 3: Convert GPS to meters (UTM) and align all matrices --

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False


def latlon_to_utm(lat, lon):
    """
    Convert WGS84 lat/lon arrays to local metric coordinates.

    Preferred: true UTM via pyproj (EPSG:32630 for Brighton area).
    Fallback : local equirectangular projection (sufficient for trajectory geometry).
    """
    lat = np.asarray(lat, dtype=np.float64)
    lon = np.asarray(lon, dtype=np.float64)

    if HAVE_PYPROJ:
        transformer = Transformer.from_crs("EPSG:4326", "EPSG:32630", always_xy=True)
        easting, northing = transformer.transform(lon, lat)
        return easting, northing

    # Fallback when pyproj is unavailable
    # Use local tangent-plane approximation in metres around trajectory centroid.
    lat0 = np.deg2rad(np.nanmean(lat))
    lon0 = np.deg2rad(np.nanmean(lon))
    latr = np.deg2rad(lat)
    lonr = np.deg2rad(lon)
    R = 6371000.0  # Earth radius [m]

    x = (lonr - lon0) * np.cos(lat0) * R
    y = (latr - np.deg2rad(np.nanmean(lat))) * R
    return x, y

if HAVE_PYPROJ:
    print("Coordinate transform: pyproj UTM (EPSG:32630)")
else:
    print("Coordinate transform: fallback local metric projection (pyproj not installed)")

# -- convert ground truth positions --
df_loc['X_m'], df_loc['Y_m'] = latlon_to_utm(df_loc['Lat'].values, df_loc['Lon'].values)
df_loc['Time_dt'] = pd.to_datetime(df_loc['Time'], unit='ms')

print(f"Ground truth time range: {df_loc['Time_dt'].min()} -> {df_loc['Time_dt'].max()}")
print(f"Feature matrix range  : {X.index.min()} -> {X.index.max()}")

# -- quick trajectory plots --
from matplotlib import pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc0 = axes[0].scatter(df_loc['Lon'], df_loc['Lat'], c=df_loc['Time'], cmap='plasma', s=2, alpha=0.5)
axes[0].set_title('GPS Trajectory (Lat/Lon)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
plt.colorbar(sc0, ax=axes[0], label='Time')

sc1 = axes[1].scatter(df_loc['X_m'], df_loc['Y_m'], c=df_loc['Time'], cmap='plasma', s=2, alpha=0.5)
axes[1].set_title('GPS Trajectory (UTM metres)')
axes[1].set_xlabel('Easting [m]'); axes[1].set_ylabel('Northing [m]')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)
plt.colorbar(sc1, ax=axes[1], label='Time')

plt.tight_layout(); plt.show()

# -- align to the same bins as feature matrices --
time_bin = '5s'
df_loc['TimeBin'] = df_loc['Time_dt'].dt.floor(time_bin)
loc_binned = df_loc.groupby('TimeBin').agg(
    X_m=('X_m', 'mean'), Y_m=('Y_m', 'mean'),
    Lat=('Lat', 'mean'), Lon=('Lon', 'mean'),
    Accuracy=('Accuracy', 'mean')
).sort_index()

common_idx = X.index.intersection(loc_binned.index)
X_aligned = X.loc[common_idx].copy()                 # dense model input
X_val_aligned = X_val.loc[common_idx].copy()         # NaN-preserving values
X_mask_aligned = X_mask.loc[common_idx].copy()       # observation mask
pos_aligned = loc_binned.loc[common_idx][['X_m', 'Y_m']].copy()
acc_aligned = loc_binned.loc[common_idx]['Accuracy']

print(f"\nAfter alignment: {len(common_idx)} time bins with both features + ground truth")
print(f"Dropped {len(X) - len(common_idx)} feature rows with no ground truth")

# -- apply GPS quality filter (now actually used) --
good_gps = (acc_aligned <= 50)
X_aligned = X_aligned.loc[good_gps]
X_val_aligned = X_val_aligned.loc[good_gps]
X_mask_aligned = X_mask_aligned.loc[good_gps]
pos_aligned = pos_aligned.loc[good_gps]
acc_aligned = acc_aligned.loc[good_gps]

print(f"GPS accuracy <=50m kept: {good_gps.sum()} / {len(good_gps)}")
print(f"Final aligned set       : {len(X_aligned)} bins")

In [ ]:
## -- Step 4: Build robust dissimilarities (RSSI + time) --

import scipy.spatial
import scipy.sparse as sp
import scipy.sparse.csgraph
import sklearn.neighbors
from scipy.stats import spearmanr
import tqdm

USE_GEODESIC = True
TIME_THRESHOLD = 15
SUBSAMPLE_STEP = 1
ENABLE_TEMPORAL_SMOOTH = True
EMA_ALPHA = 0.35
OVERLAP_PENALTY = 0.15
COMPUTE_GEO_FUSED = True
COMPUTE_GEO_SIGNAL = True
TRAIN_DISTANCE_SOURCE = "fused"

X_sub = X_aligned.values[::SUBSAMPLE_STEP].astype(np.float32)
X_val_sub = X_val_aligned.values[::SUBSAMPLE_STEP].astype(np.float32)
X_mask_sub = X_mask_aligned.values[::SUBSAMPLE_STEP].astype(np.float32)
pos_sub = pos_aligned.values[::SUBSAMPLE_STEP].astype(np.float32)
time_sec = (X_aligned.index[::SUBSAMPLE_STEP] - X_aligned.index[0]).total_seconds().values.astype(np.float32)

N, F = X_sub.shape
print(f"Working set: N={N} samples, F={F} features")
print(f"Position array: {pos_sub.shape}")
print(f"Observed ratio: {X_mask_sub.mean():.3f}")

if ENABLE_TEMPORAL_SMOOTH:
    val_df = pd.DataFrame(X_val_sub)
    mask_df = pd.DataFrame(X_mask_sub)
    smoothed = val_df.ffill().bfill().ewm(alpha=EMA_ALPHA, adjust=False).mean()
    smoothed = smoothed.where(mask_df > 0)
    X_val_proc = smoothed.values.astype(np.float32)
    print(f"Applied EMA smoothing on observed RSSI (alpha={EMA_ALPHA})")
else:
    X_val_proc = X_val_sub

D_gt = scipy.spatial.distance_matrix(pos_sub, pos_sub).astype(np.float32)


def _overlap_ratio(mask):
    common = mask @ mask.T
    min_obs = np.minimum(mask.sum(axis=1, keepdims=True), mask.sum(axis=1, keepdims=True).T)
    return common / (min_obs + 1e-6)


def cosine_centered_distance_nan(X_val_nan, X_mask, overlap_penalty=0.1):
    X0 = np.nan_to_num(X_val_nan, nan=0.0)
    counts = np.maximum(X_mask.sum(axis=1, keepdims=True), 1.0)
    means = (X0 * X_mask).sum(axis=1, keepdims=True) / counts
    Xc = (X0 - means) * X_mask

    norms = np.linalg.norm(Xc, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-6)
    cos_sim = (Xc @ Xc.T) / (norms @ norms.T)
    cos_sim = np.clip(cos_sim, -1.0, 1.0)

    ov = _overlap_ratio(X_mask)
    D = (1.0 - cos_sim) + overlap_penalty * (1.0 - ov)
    np.fill_diagonal(D, 0.0)
    return D.astype(np.float32)


def weighted_euclidean_nan(X_val_nan, X_mask, overlap_penalty=0.1, eps=1e-6):
    var = np.nanvar(X_val_nan, axis=0)
    w = 1.0 / (var + eps)
    w = w / (np.mean(w) + eps)

    X0 = np.nan_to_num(X_val_nan, nan=0.0)
    M = X_mask

    Xw = X0 * M * np.sqrt(w)
    X2w = (X0 ** 2) * M * w

    term1 = X2w @ M.T
    term2 = term1.T
    cross = Xw @ Xw.T
    common_w = (M * w) @ M.T

    dist2 = np.maximum(term1 + term2 - 2.0 * cross, 0.0)
    dist2 = dist2 / (common_w + 1e-6)

    ov = _overlap_ratio(M)
    D = np.sqrt(dist2) + overlap_penalty * (1.0 - ov)
    np.fill_diagonal(D, 0.0)
    return D.astype(np.float32)


def _monotonic_score(D_metric, D_true, step=12):
    d = D_metric[::step, ::step]
    g = D_true[::step, ::step]
    iu = np.triu_indices(d.shape[0], k=1)
    corr = spearmanr(d[iu], g[iu]).correlation
    return float(corr)


print("\nComputing candidate RSSI dissimilarities...")
D_cos = cosine_centered_distance_nan(X_val_proc, X_mask_sub, overlap_penalty=OVERLAP_PENALTY)
D_wl2 = weighted_euclidean_nan(X_val_proc, X_mask_sub, overlap_penalty=OVERLAP_PENALTY)

D_signal_candidates = {
    "cosine_centered_overlap": D_cos,
    "weighted_l2_overlap": D_wl2,
}

metric_scores = {name: _monotonic_score(Dm, D_gt) for name, Dm in D_signal_candidates.items()}
for name, score in metric_scores.items():
    print(f"  {name:24s} Spearman vs GT = {score:.4f}")

D_signal_name = max(metric_scores, key=metric_scores.get)
D_signal = D_signal_candidates[D_signal_name]
print(f"Selected D_signal: {D_signal_name}")
print(f"D_signal range: [{D_signal.min():.3f}, {D_signal.max():.3f}]")

print("\nComputing time dissimilarity matrix...")
D_time = np.abs(np.subtract.outer(time_sec, time_sec)).astype(np.float32)
print(f"D_time range: [{D_time.min():.1f}, {D_time.max():.1f}] sec")


In [ ]:
## ── Step 4b.1: Speed diagnostics (transport-mode visualization) ───────
# Compute instantaneous speed from consecutive ground-truth positions (m/s)
# and plot distribution + time series so we can see walking vs vehicular segments.

time_diffs = np.diff(time_sec)            # seconds between consecutive samples
pos_diffs  = np.linalg.norm(np.diff(pos_sub, axis=0), axis=1)  # metres
speeds = pos_diffs / (time_diffs + 1e-9)  # m/s

print(f"Instantaneous speed: median={np.median(speeds):.2f} m/s, mean={np.mean(speeds):.2f} m/s")
print("Speed percentiles (m/s):", np.round(np.percentile(speeds, [10,25,50,75,90,99]), 2))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))

# Histogram
ax[0].hist(speeds, bins=100, range=(0, max(5, speeds.max())), color="#4C72B0", alpha=0.8)
ax[0].set_xlabel('Speed [m/s]'); ax[0].set_ylabel('Count')
ax[0].set_title('Instantaneous speed distribution')
ax[0].axvline(1.4, color='r', ls='--', label='walk ~1.4 m/s')
ax[0].axvline(3.0, color='orange', ls='--', label='run/bike ~3 m/s')
ax[0].axvline(8.0, color='g', ls='--', label='vehicular ~8 m/s')
ax[0].legend(fontsize=8)

# Speed over time
ax[1].scatter(time_sec[:-1], speeds, s=2, alpha=0.6)
ax[1].set_xlabel('Time [s]'); ax[1].set_ylabel('Speed [m/s]')
ax[1].set_title('Speed vs time')
ax[1].set_ylim(0, max(5, speeds.max()))

plt.tight_layout(); plt.show()

# Simple segmentation counts (useful for adaptive fusion later)
walk_frac = np.mean(speeds < 2.0)
veh_frac  = np.mean(speeds >= 2.0)
print(f"Estimated walk fraction: {walk_frac:.2%}, vehicular fraction: {veh_frac:.2%}")

# Save speeds for possible adaptive fusion later
instant_speeds = speeds  # available in notebook namespace

In [ ]:
## -- Step 4c: Correlation-driven smooth fusion --

from matplotlib import pyplot as plt


def sample_pairs(D_metric, D_gt, step=12):
    d = D_metric[::step, ::step]
    g = D_gt[::step, ::step]
    iu = np.triu_indices(d.shape[0], k=1)
    x = d[iu].astype(np.float64)
    y = g[iu].astype(np.float64)
    keep = np.isfinite(x) & np.isfinite(y)
    return x[keep], y[keep]


def fuse_dissimilarities_smooth(D_signal, D_time, time_threshold=15,
                                alpha=0.82, beta=1.0, tau=None,
                                min_pairs=2000):
    close_mask = (D_time > 0) & (D_time <= time_threshold)
    if close_mask.sum() < min_pairs:
        nonzero = D_time[D_time > 0]
        for p in [5, 10, 20, 30, 40, 50, 60]:
            t = np.percentile(nonzero, p)
            candidate = (D_time > 0) & (D_time <= t)
            if candidate.sum() >= min_pairs:
                close_mask = candidate
                print(f"fusion: expanded close-pair threshold to {t:.1f}s (p={p}%)")
                break

    if close_mask.sum() == 0:
        raise RuntimeError("No close-in-time pairs available for fusion scaling.")

    sig_close = D_signal[close_mask]
    time_close = D_time[close_mask]

    sig_scale = np.median(sig_close[np.isfinite(sig_close)])
    time_scale = np.median(time_close[np.isfinite(time_close)])

    Dsig_n = D_signal / (sig_scale + 1e-6)
    Dtime_n = D_time / (time_scale + 1e-6)

    if tau is None:
        D_fused = alpha * Dsig_n + (1.0 - alpha) * beta * Dtime_n
        fusion_mode = "convex"
    else:
        A = np.exp(-Dsig_n / tau)
        B = np.exp(-(beta * Dtime_n) / tau)
        D_fused = -tau * np.log(A + B + 1e-12)
        fusion_mode = f"softmin(tau={tau})"

    np.fill_diagonal(D_fused, 0.0)

    stats = {
        "close_pairs": int(close_mask.sum()),
        "sig_scale": float(sig_scale),
        "time_scale": float(time_scale),
        "alpha": float(alpha),
        "beta": float(beta),
        "mode": fusion_mode,
    }
    return D_fused.astype(np.float32), stats, close_mask, Dsig_n.astype(np.float32), Dtime_n.astype(np.float32)


_, fusion_stats0, close_pairs_mask, Dsig_n, Dtime_n = fuse_dissimilarities_smooth(
    D_signal,
    D_time,
    time_threshold=TIME_THRESHOLD,
    alpha=0.82,
    beta=1.0,
    tau=None,
    min_pairs=5000,
)

alpha_list = [0.60, 0.70, 0.78, 0.82, 0.86, 0.90]
alpha_rows = []
for a in alpha_list:
    D_fused_a = a * Dsig_n + (1.0 - a) * Dtime_n
    np.fill_diagonal(D_fused_a, 0.0)
    x, y = sample_pairs(D_fused_a, D_gt, step=12)
    rho = spearmanr(x, y).correlation
    alpha_rows.append((a, float(rho)))

alpha_best, alpha_best_rho = max(alpha_rows, key=lambda t: t[1])
D_fused = (alpha_best * Dsig_n + (1.0 - alpha_best) * Dtime_n).astype(np.float32)
np.fill_diagonal(D_fused, 0.0)

fusion_stats = dict(fusion_stats0)
fusion_stats["alpha"] = float(alpha_best)
fusion_stats["mode"] = "convex(alpha-sweep)"

print("Fusion stats:")
for k, v in fusion_stats.items():
    print(f"  {k:12s}: {v}")
print("Alpha sweep:")
for a, rho in alpha_rows:
    print(f"  alpha={a:.2f}  spearman={rho:.4f}")
print(f"alpha_best={alpha_best:.2f}  spearman={alpha_best_rho:.4f}")
print(f"D_fused range: [{D_fused.min():.3f}, {D_fused.max():.3f}]")

In [ ]:
## -- Step 4d: Geodesic variants and training source selection --

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra


def build_mutual_knn_graph(dm, n_neighbors=50):
    n = dm.shape[0]
    nbrs = sklearn.neighbors.NearestNeighbors(
        n_neighbors=n_neighbors, metric="precomputed", n_jobs=-1
    ).fit(dm)
    dist, idx = nbrs.kneighbors(dm, return_distance=True)

    rows = np.repeat(np.arange(n), n_neighbors)
    cols = idx.reshape(-1)
    vals = dist.reshape(-1)

    G = csr_matrix((vals, (rows, cols)), shape=(n, n))
    A = G.copy()
    A.data = np.ones_like(A.data)
    mutual = A.multiply(A.T)
    W = G.maximum(G.T).multiply(mutual)

    chain_i = np.arange(n - 1)
    chain_j = chain_i + 1
    chain_w = dm[chain_i, chain_j]
    T = csr_matrix(
        (np.r_[chain_w, chain_w], (np.r_[chain_i, chain_j], np.r_[chain_j, chain_i])),
        shape=(n, n)
    )
    W = W.maximum(T)
    W.eliminate_zeros()
    return W


def get_geodesic_dissimilarity_matrix(dm, k_list=(30, 50, 80, 100, 130), name=""):
    info = {
        "name": name,
        "connected": False,
        "used_k": None,
        "n_components": None,
        "inf_fraction": None,
    }

    best_graph = None
    best_components = None

    print(f"\n{name} geodesic build:")
    for k in k_list:
        W = build_mutual_knn_graph(dm, n_neighbors=k)
        n_comp, _ = connected_components(W, directed=False)
        print(f"  k={k:3d}: connected components = {n_comp}")
        best_graph = W
        best_components = n_comp
        if n_comp == 1:
            info["connected"] = True
            info["used_k"] = int(k)
            info["n_components"] = int(n_comp)
            break

    if not info["connected"]:
        info["used_k"] = int(k_list[-1])
        info["n_components"] = int(best_components)
        return None, info

    D_geo = dijkstra(best_graph, directed=False).astype(np.float32)
    inf_fraction = float(np.mean(~np.isfinite(D_geo)))
    info["inf_fraction"] = inf_fraction

    finite_vals = D_geo[np.isfinite(D_geo)]
    plt.figure(figsize=(6, 3))
    plt.hist(finite_vals, bins=120, alpha=0.85)
    plt.title(f"Histogram of finite geodesic distances ({name})")
    plt.xlabel("D_geo (finite)")
    plt.ylabel("Count")
    plt.tight_layout(); plt.show()

    if inf_fraction > 0:
        finite_mask = np.isfinite(D_geo)
        max_finite = np.max(D_geo[finite_mask])
        D_geo = D_geo.copy()
        D_geo[~finite_mask] = 10.0 * max_finite

    return D_geo, info


D_geo_fused, geo_info_fused = None, None
D_geo_signal, geo_info_signal = None, None

if USE_GEODESIC and COMPUTE_GEO_FUSED:
    D_geo_fused, geo_info_fused = get_geodesic_dissimilarity_matrix(D_fused, name="fused")

if USE_GEODESIC and COMPUTE_GEO_SIGNAL:
    D_geo_signal, geo_info_signal = get_geodesic_dissimilarity_matrix(D_signal, name="signal")

cases = {
    "signal": D_signal,
    "time": D_time,
    "fused": D_fused,
    "geo_fused": D_geo_fused,
    "geo_signal": D_geo_signal,
}
cases = {k: v for k, v in cases.items() if v is not None}

if TRAIN_DISTANCE_SOURCE not in cases:
    print(f"TRAIN_DISTANCE_SOURCE '{TRAIN_DISTANCE_SOURCE}' unavailable, fallback to 'fused'")
    TRAIN_DISTANCE_SOURCE = "fused"

D_train = cases[TRAIN_DISTANCE_SOURCE]
GEODESIC_USED = TRAIN_DISTANCE_SOURCE.startswith("geo_")
D_geo = D_geo_fused if D_geo_fused is not None else D_geo_signal
geo_info = geo_info_fused if geo_info_fused is not None else geo_info_signal

print("\nTraining source selection:")
print(f"  TRAIN_DISTANCE_SOURCE = {TRAIN_DISTANCE_SOURCE}")
print(f"  D_train range         = [{np.nanmin(D_train):.3f}, {np.nanmax(D_train):.3f}]")
print(f"  cases                 = {list(cases.keys())}")

In [ ]:
## -- Step 4e: Correlation-first evaluation and calibrated plots --

def eval_corr(D_metric, D_gt, step=12, name=""):
    d = D_metric[::step, ::step]
    g = D_gt[::step, ::step]
    iu = np.triu_indices(d.shape[0], k=1)
    x = d[iu].astype(np.float64)
    y = g[iu].astype(np.float64)
    keep = np.isfinite(x) & np.isfinite(y)
    x = x[keep]
    y = y[keep]
    rho = spearmanr(x, y).correlation
    r = np.corrcoef(x, y)[0, 1]
    return float(rho), float(r)


def fit_affine(D_metric, D_gt, step=12):
    d = D_metric[::step, ::step]
    g = D_gt[::step, ::step]
    iu = np.triu_indices(d.shape[0], k=1)
    x = d[iu].astype(np.float64)
    y = g[iu].astype(np.float64)
    keep = np.isfinite(x) & np.isfinite(y)
    x = x[keep]
    y = y[keep]
    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    if a < 0:
        a = 0.0
    return float(a), float(b)


def binned_curve(y_true, y_pred, bins=150):
    edges = np.linspace(0, np.max(y_true), bins)
    idx = np.digitize(y_true, edges)
    med = np.full(len(edges) - 1, np.nan)
    q25 = np.full(len(edges) - 1, np.nan)
    q75 = np.full(len(edges) - 1, np.nan)
    for i in range(1, len(edges)):
        vals = y_pred[idx == i]
        if vals.size > 0:
            q25[i - 1], med[i - 1], q75[i - 1] = np.percentile(vals, [25, 50, 75])
    return edges[:-1], med, q25, q75


def plot_raw_case(D_metric, D_gt, name, ax, step=12):
    d = D_metric[::step, ::step]
    g = D_gt[::step, ::step]
    iu = np.triu_indices(d.shape[0], k=1)
    x = d[iu].astype(np.float64)
    y = g[iu].astype(np.float64)
    keep = np.isfinite(x) & np.isfinite(y)
    x = x[keep]
    y = y[keep]
    xx, med, q25, q75 = binned_curve(y, x)
    ax.plot(xx, med, lw=1.8)
    ax.fill_between(xx, q25, q75, alpha=0.22)
    ax.set_title(name)
    ax.set_xlabel("True Distance [m]")
    ax.set_ylabel("Dissimilarity")


def plot_calibrated_case(D_metric, D_gt, name, ax, step=12):
    d = D_metric[::step, ::step]
    g = D_gt[::step, ::step]
    iu = np.triu_indices(d.shape[0], k=1)
    x = d[iu].astype(np.float64)
    y = g[iu].astype(np.float64)
    keep = np.isfinite(x) & np.isfinite(y)
    x = x[keep]
    y = y[keep]
    rho, r = eval_corr(D_metric, D_gt, step=step, name=name)
    a, b = fit_affine(D_metric, D_gt, step=step)
    y_pred = a * x + b
    xx, med, q25, q75 = binned_curve(y, y_pred)
    lim = np.nanmax(xx)
    ax.plot(xx, med, lw=1.8)
    ax.fill_between(xx, q25, q75, alpha=0.22)
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_title(f"{name} | rho={rho:.3f}, r={r:.3f}")
    ax.set_xlabel("True Distance [m]")
    ax.set_ylabel("Predicted Distance [m]")
    return rho, r, a, b


case_names = list(cases.keys())

fig, axes = plt.subplots(1, len(case_names), figsize=(5 * len(case_names), 4))
if len(case_names) == 1:
    axes = [axes]
for ax, name in zip(axes, case_names):
    plot_raw_case(cases[name], D_gt, name, ax, step=12)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(case_names), figsize=(5 * len(case_names), 4))
if len(case_names) == 1:
    axes = [axes]

rows = []
for ax, name in zip(axes, case_names):
    rho, r, a, b = plot_calibrated_case(cases[name], D_gt, name, ax, step=12)
    rows.append((name, rho, r, a, b))
plt.tight_layout(); plt.show()

metrics_df = pd.DataFrame(rows, columns=["name", "spearman", "pearson", "a", "b"]).sort_values("spearman", ascending=False)
print(metrics_df.to_string(index=False))
BEST_CASE_BY_SPEARMAN = metrics_df.iloc[0]["name"]
print(f"BEST_CASE_BY_SPEARMAN = {BEST_CASE_BY_SPEARMAN}")

In [ ]:
## -- Step 4f: MDS baseline on D_train --

from sklearn.manifold import MDS

NTH_ISOMAP = 10
D_train_sub = D_train[::NTH_ISOMAP, ::NTH_ISOMAP]
pos_sub_isomap = pos_sub[::NTH_ISOMAP]

print(f"Running MDS on {D_train_sub.shape[0]} points ...")
embedding_isomap = MDS(
    metric=True, dissimilarity='precomputed',
    max_iter=200, normalized_stress=False, n_init=4, random_state=42
)
proj_isomap = embedding_isomap.fit_transform(D_train_sub)
print(f"  MDS stress = {embedding_isomap.stress_:.2f}")

# ── Plot: ground truth vs Isomap ──
def plot_colorized(positions, gt_positions, title=None, show=True, alpha=1.0, ax=None):
    """Scatter coloured by ground-truth position for topology comparison."""
    norm = lambda d: (d - d.min()) / (d.max() - d.min() + 1e-9)
    center = 0.5 * (gt_positions.min(axis=0) + gt_positions.max(axis=0))
    rgb = np.zeros((len(gt_positions), 3))
    rgb[:, 0] = 1 - 0.9 * norm(gt_positions[:, 0])
    rgb[:, 1] = 0.8 * norm(np.square(np.linalg.norm(gt_positions - center, axis=1)))
    rgb[:, 2] = 0.9 * norm(gt_positions[:, 1])
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    if title:
        ax.set_title(title, fontsize=14)
    ax.scatter(positions[:, 0], positions[:, 1], c=rgb, s=12, alpha=alpha, linewidths=0)
    ax.set_aspect('equal')
    if show and ax is None:
        plt.show()
    return ax

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_colorized(pos_sub_isomap, pos_sub_isomap,
               title="Ground Truth (subsampled)", show=False, ax=axes[0])
axes[0].set_xlabel("Easting [m]"); axes[0].set_ylabel("Northing [m]")
plot_colorized(proj_isomap, pos_sub_isomap,
               title="MDS Baseline", show=False, ax=axes[1])
axes[1].set_xlabel("dim 1"); axes[1].set_ylabel("dim 2")
plt.tight_layout(); plt.show()

In [ ]:
## ── Step 5: Siamese Network for Channel Charting ─────────────────────────

import os
import tensorflow as tf
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ── 5a. Build the Forward Charting Function ──
def build_charting_function(n_features, name="ForwardChartingFunction"):
    """Maps an RSSI fingerprint vector → 2D channel chart embedding."""
    inp = tf.keras.Input(shape=(n_features,), dtype=tf.float32, name="input")
    x = tf.keras.layers.Dense(512, activation="relu")(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(2, activation="linear")(x)
    return tf.keras.Model(inputs=inp, outputs=x, name=name)

cc_model = build_charting_function(F)
cc_model.summary()

# ── 5b. Build the Siamese wrapper ──
input_A = tf.keras.Input(shape=(F,), dtype=tf.float32)
input_B = tf.keras.Input(shape=(F,), dtype=tf.float32)
emb_A = cc_model(input_A)
emb_B = cc_model(input_B)
output = tf.keras.layers.concatenate([emb_A, emb_B], axis=1)
siamese = tf.keras.Model([input_A, input_B], output, name="SiameseNetwork")

# ── 5c. Siamese loss (stress minimisation) ──
dissimilarity_margin = float(np.quantile(D_train, 0.01))

def siamese_loss(y_true, y_pred):
    y_true = y_true[:, 0]
    pos_A, pos_B = y_pred[:, :2], y_pred[:, 2:]
    d_pred = tf.math.reduce_euclidean_norm(pos_A - pos_B, axis=1)
    return tf.reduce_mean(tf.square(d_pred - y_true) / (y_true + dissimilarity_margin))

# ── 5d. Build random-pair dataset over ALL data ──
X_tensor = tf.constant(X_sub, dtype=tf.float32)
D_train_tensor = tf.constant(D_train, dtype=tf.float32)
n_samples = tf.shape(X_tensor)[0].numpy()

pair_ds = tf.data.Dataset.zip((tf.data.Dataset.random(), tf.data.Dataset.random()))

@tf.function
def fill_pair(rA, rB):
    iA = rA % n_samples
    iB = rB % n_samples
    return (X_tensor[iA], X_tensor[iB]), D_train_tensor[iA, iB]

pair_ds = pair_ds.map(fill_pair)

print(f"\n✓ Siamese network ready.  N={n_samples}, F={F}")
print(f"  margin ε = {dissimilarity_margin:.4f}")
print(f"  Training on source: {TRAIN_DISTANCE_SOURCE}")

In [ ]:
## ── Step 6: Training Loop ────────────────────────────────────────────────

optimizer = tf.keras.optimizers.Adam()
siamese.compile(loss=siamese_loss, optimizer=optimizer)

# Training schedule following the DICHASUS tutorial:
#   - More samples per session for better convergence
#   - Gradually increasing batch sizes (coarse → fine detail)
samples_per_session = 150_000
learning_rates = [1e-2, 1e-2, 8e-3, 4e-3, 1e-3, 5e-4, 2e-4, 1e-4]
batch_sizes    = [500,  1000, 1500, 2000, 3000, 4000, 5000, 6000]

for s, (lr, bs) in enumerate(zip(learning_rates, batch_sizes)):
    print(f"\n── Session {s+1}/{len(learning_rates)}  "
          f"lr={lr:.0e}  batch={bs} ──")
    optimizer.learning_rate.assign(lr)
    siamese.fit(
        pair_ds.batch(bs).prefetch(tf.data.AUTOTUNE),
        steps_per_epoch=samples_per_session // bs,
        verbose=1
    )

In [ ]:
## ── Step 7: Channel Charting Evaluation Metrics ──────────────────────────
#
# Standard CC quality metrics (no localization / no affine transform):
#   - Kruskal Stress  (how well pairwise dissimilarities are preserved)
#   - Trustworthiness (are CC neighbours actually close in dissimilarity space?)
#   - Continuity      (are dissimilarity neighbours close in the CC?)

from sklearn.manifold import trustworthiness as sklearn_tw

# ── Infer channel chart positions on ALL data ──
cc_positions = cc_model.predict(X_sub, batch_size=2048, verbose=0)

# ── Pairwise distances in the channel chart ──
D_cc = scipy.spatial.distance_matrix(cc_positions, cc_positions).astype(np.float32)

# ─────────────────────────────────────────────────────────────────────────
# 7a.  Kruskal Stress (type 1)
# ─────────────────────────────────────────────────────────────────────────
iu = np.triu_indices(N, k=1)  # upper triangle, no diagonal
d_train_flat = D_train[iu]
d_cc_flat  = D_cc[iu]

# scale-normalise (CC output is in arbitrary units)
scale = d_train_flat.mean() / (d_cc_flat.mean() + 1e-12)
d_cc_scaled = d_cc_flat * scale

kruskal_stress = np.sqrt(
    np.sum((d_train_flat - d_cc_scaled) ** 2) / np.sum(d_train_flat ** 2)
)
print(f"Kruskal Stress (type 1) = {kruskal_stress:.4f}   (lower is better, 0 = perfect)")

# ─────────────────────────────────────────────────────────────────────────
# 7b.  Trustworthiness
# ─────────────────────────────────────────────────────────────────────────
for k in [5, 10, 20]:
    tw = sklearn_tw(D_train, cc_positions, n_neighbors=k, metric='precomputed')
    print(f"Trustworthiness  (k={k:2d}) = {tw:.4f}   (higher is better, 1 = perfect)")

# ─────────────────────────────────────────────────────────────────────────
# 7c.  Continuity  (dual of TW)
# ─────────────────────────────────────────────────────────────────────────
def continuity(D_high, X_low, n_neighbors=10):
    """Continuity metric — dual of trustworthiness."""
    n = D_high.shape[0]
    D_low = scipy.spatial.distance_matrix(X_low, X_low)
    
    # ranks in high-dim  (dissimilarity space)
    nn_high = np.argsort(D_high, axis=1)[:, 1:n_neighbors+1]
    # ranks in low-dim   (CC space)
    ranks_low = np.argsort(np.argsort(D_low, axis=1), axis=1)
    
    violations = 0.0
    for i in range(n):
        for j in nn_high[i]:
            r = ranks_low[i, j]
            if r > n_neighbors:
                violations += r - n_neighbors
    
    # normalise
    denom = n * n_neighbors * (2 * n - 3 * n_neighbors - 1) / 2.0
    if denom == 0:
        return 1.0
    return 1.0 - violations / denom

for k in [5, 10, 20]:
    co = continuity(D_train, cc_positions, n_neighbors=k)
    print(f"Continuity       (k={k:2d}) = {co:.4f}   (higher is better, 1 = perfect)")

Z = cc_positions
N = Z.shape[0]
k_tutorial = int(round(0.05 * N))
k_tutorial = max(5, min(k_tutorial, N - 1))
TW_tutorial = sklearn_tw(D_train, Z, n_neighbors=k_tutorial, metric='precomputed')
CO_tutorial = continuity(D_train, Z, n_neighbors=k_tutorial)
print(f"TW(k_tutorial= {k_tutorial}) = {TW_tutorial:.4f}    CO(k_tutorial= {k_tutorial}) = {CO_tutorial:.4f}")

In [ ]:
## -- Step 8: Channel chart visualization + change summary --

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Left: ground truth positions (UTM)
plot_colorized(pos_sub, pos_sub, title="Ground Truth (UTM)", show=False, ax=axes[0])
axes[0].set_xlabel("Easting [m]"); axes[0].set_ylabel("Northing [m]")

# Middle: Isomap / MDS baseline (subsampled)
plot_colorized(proj_isomap, pos_sub_isomap,
               title="Isomap / MDS Baseline", show=False, ax=axes[1])
axes[1].set_xlabel("dim 1"); axes[1].set_ylabel("dim 2")

# Right: learned channel chart (arbitrary coordinates)
plot_colorized(cc_positions, pos_sub, title="Siamese Channel Chart", show=False, ax=axes[2])
axes[2].set_xlabel("CC dim 1"); axes[2].set_ylabel("CC dim 2")

plt.suptitle(
    f"Kruskal Stress = {kruskal_stress:.4f}  |  "
    f"Geodesic={'ON' if GEODESIC_USED else 'OFF'}  |  "
    f"TIME_THRESHOLD={TIME_THRESHOLD}s",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('cc_channel_chart.png', dpi=200, bbox_inches='tight')
plt.show()

# -- Shepard diagram: D_train vs D_cc --
fig, ax = plt.subplots(figsize=(5, 5))
subsample = np.random.choice(len(d_train_flat), size=min(50000, len(d_train_flat)), replace=False)
ax.scatter(d_train_flat[subsample], d_cc_scaled[subsample], s=1, alpha=0.1, rasterized=True)
ax.plot([0, d_train_flat.max()], [0, d_train_flat.max()], 'r--', lw=1, label='ideal')
ax.set_xlabel("Training dissimilarity")
ax.set_ylabel("CC distance (scaled)")
ax.set_title("Shepard Diagram")
ax.legend()
plt.tight_layout()
plt.savefig('cc_shepard_diagram.png', dpi=200, bbox_inches='tight')
plt.show()

print("\n" + "=" * 80)
print("  Channel Charting Quality Summary")
print("=" * 80)
print(f"  SIGNAL_METRIC          = {D_signal_name}")
print(f"  TIME_THRESHOLD         = {TIME_THRESHOLD}")
print(f"  FUSION_MODE            = {fusion_stats['mode']}")
print(f"  alpha_best             = {fusion_stats['alpha']:.2f}")
print(f"  TRAIN_DISTANCE_SOURCE  = {TRAIN_DISTANCE_SOURCE}")
print(f"  BEST_CASE_BY_SPEARMAN  = {BEST_CASE_BY_SPEARMAN}")
if geo_info_fused is not None:
    print(f"  GEO_FUSED_COMPONENTS   = {geo_info_fused['n_components']}  k={geo_info_fused['used_k']}  inf={geo_info_fused['inf_fraction']}")
if geo_info_signal is not None:
    print(f"  GEO_SIGNAL_COMPONENTS  = {geo_info_signal['n_components']}  k={geo_info_signal['used_k']}  inf={geo_info_signal['inf_fraction']}")
print(f"  Kruskal Stress         = {kruskal_stress:.4f}")
for k in [5, 10, 20]:
    tw = sklearn_tw(D_train, cc_positions, n_neighbors=k, metric='precomputed')
    co = continuity(D_train, cc_positions, n_neighbors=k)
    print(f"  TW(k={k:2d}) = {tw:.4f}    CO(k={k:2d}) = {co:.4f}")
print(f"TW(k_tutorial= {k_tutorial}) = {TW_tutorial:.4f}    CO(k_tutorial= {k_tutorial}) = {CO_tutorial:.4f}")
print("=" * 80)
print("\nCase ranking (Spearman desc):")
print(metrics_df.to_string(index=False))